# 💪 Physical State Classification Model

This notebook implements a text classification model using a pretrained transformer (XLM-RoBERTa) to predict physical states from textual data.

The model includes class imbalance handling using weighted loss.

In [1]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from torch.nn import CrossEntropyLoss

c:\Users\a7mda\OneDrive\Desktop\emotion-physical-classifier\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 📂 Dataset Description

The dataset contains textual data labeled with physical states.

- Input: text
- Output: physical_state label

Missing values are removed to ensure data quality before training.

In [2]:
df = pd.read_csv("../datasets/data.csv")
df = df.dropna()

print(df["physical_state"].value_counts())

physical_state
tired        30
energetic    28
weak         23
tense        21
relaxed      18
Name: count, dtype: int64


## 🧹 Data Cleaning & Label Encoding

The preprocessing steps include:

- Removing missing values
- Encoding physical state labels into numerical values

This ensures the model can process the data efficiently.

In [3]:
physical_labels = sorted(df["physical_state"].unique())
physical2id = {label: i for i, label in enumerate(physical_labels)}
id2physical = {i: label for label, i in physical2id.items()}

df["label"] = df["physical_state"].map(physical2id)

## 🔀 Data Splitting

The dataset is divided into:

- Training set (70%)
- Validation set (15%)
- Test set (15%)

This allows proper evaluation on unseen data and prevents overfitting.

In [4]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5)

## 🔄 Dataset Conversion

The pandas DataFrame is converted into HuggingFace Dataset format.

This step is necessary to use the HuggingFace training pipeline efficiently.

In [5]:
train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
val_dataset = Dataset.from_pandas(val_df[["text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

## 🔤 Tokenization

Text data is tokenized using the XLM-RoBERTa tokenizer.

- Converts text into numerical tokens
- Applies padding and truncation

This prepares the input for the transformer model.

In [6]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding=True)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 18/18 [00:00<00:00, 4880.25 examples/s]


## 🤖 Model Architecture

We use a pretrained transformer model (XLM-RoBERTa) for classification.

- Backbone: XLM-RoBERTa
- Task: Multi-class classification
- Output: predicted physical state

The classification head is fine-tuned on our dataset.

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=len(physical_labels),
    id2label=id2physical,
    label2id=physical2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4396.38it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## ⚙️ Training Configuration

The model is trained using:

- Learning rate: 2e-5
- Batch size: 8
- Epochs: 15
- Evaluation at each epoch

The best model is selected based on F1-score.

In [11]:
training_args = TrainingArguments(
    output_dir="./results_physical",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=10
)

## ⚖️ Class Imbalance Handling

To address class imbalance, we compute class weights and apply them in the loss function.

This ensures that minority classes are not ignored during training and improves overall performance.

In [12]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(list(range(len(physical_labels)))),
    y=df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

## 🧠 Custom Trainer

A custom Trainer is implemented to integrate weighted cross-entropy loss.

This allows the model to account for class imbalance during training.

In [13]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

## 📊 Evaluation Metrics

We evaluate the model using:

- **Accuracy**: Measures overall correctness
- **F1-score (weighted)**: Balances precision and recall, especially important for imbalanced datasets

These metrics provide a comprehensive evaluation of model performance.

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)
    
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

## 🏋️ Model Training

The model is trained on the training dataset and evaluated on the validation set at each epoch.

Training progress includes loss reduction and metric improvements.

In [15]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [17]:
trainer.train()

c:\Users\a7mda\OneDrive\Desktop\emotion-physical-classifier\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.597975,1.677463,0.111111,0.043573
2,1.620831,1.673657,0.055556,0.006536
3,1.593410,1.588898,0.111111,0.084722
4,1.508296,1.569498,0.333333,0.303086
5,1.447138,1.510975,0.277778,0.256481
6,1.303544,1.308433,0.666667,0.670966
7,1.189350,1.165120,0.555556,0.560185
8,1.036517,1.004691,0.666667,0.679453
9,0.907507,1.025137,0.666667,0.679453
10,0.676232,0.738635,0.666667,0.678029


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.38s/it]
c:\Users\a7mda\OneDrive\Desktop\emotion-physical-classifier\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.65s/it]
c:\Users\a7mda\OneDrive\Desktop\emotion-physical-classifier\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.45s/it]
c:\Users\a7mda\OneDrive\Desktop\emotion-physical-classifier\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader

TrainOutput(global_step=165, training_loss=0.9733217123782996, metrics={'train_runtime': 299.5136, 'train_samples_per_second': 4.207, 'train_steps_per_second': 0.551, 'total_flos': 9065242252080.0, 'train_loss': 0.9733217123782996, 'epoch': 15.0})

In [18]:
results = trainer.evaluate(test_dataset)
print(results)

c:\Users\a7mda\OneDrive\Desktop\emotion-physical-classifier\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.373217,0.619182,15,0.722222,0.721176


{'eval_loss': 0.6191821694374084, 'eval_accuracy': 0.7222222222222222, 'eval_f1': 0.7211760461760461}


## 📊 Final Test Results

The model achieved:

- Accuracy: 72%
- F1-score: 71%

The results demonstrate strong performance and effective handling of class imbalance.

In [19]:
model.save_pretrained("physical_model")
tokenizer.save_pretrained("physical_model")

print("✅ Physical model saved!")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.63s/it]

✅ Physical model saved!


## 🧠 Discussion

The model performs well due to:

- Use of transfer learning (XLM-RoBERTa)
- Weighted loss for class imbalance
- Proper training strategy

The results indicate stable learning and good generalization.